# dki-agents 端到端演示：数据 → 知识 → 洞察

本 Notebook 串起赛题三个任务，形成「数据 → 知识 → 洞察」完整闭环，评审可交互复现：

1. **任务一 数据处理**：合成含噪病历 → DataMate 算子流水线清洗
2. **任务二 知识图谱 + 问答**：实体/关系抽取 → 三元组构建校验 → 图谱 → 模板问答；抽取质量 vs 金标 F1
3. **NER 微调（任务一加分）**：dict 远程监督弱标注微调 bert-base-chinese，三后端实体 F1 对比
4. **任务三 数据分析 + BI**：NL2SQL 查询 → 指标计算 → ECharts 图文报告内联
5. **NPU 算子优化（任务二加分）**：FP16 vs INT8 昇腾 910B 吞吐/能效/精度对比

> 内核：项目 `.venv`。所有数字均由真实代码运行产生，与 CLI(`python -m dki ...`) 同源。

## 0. 环境自检与项目路径

In [ ]:
import os, sys, json
os.environ.setdefault("PYTHONUTF8", "1")
from pathlib import Path

# 定位项目根（notebooks/ 的上一层），加入 sys.path
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dki.paths import RAW_DATA_DIR, OUTPUT_DIR, GOLD_DATA_DIR, PROJECT_ROOT
print("项目根:", PROJECT_ROOT)
print("原始数据:", RAW_DATA_DIR, "| 存在", RAW_DATA_DIR.exists())
print("输出目录:", OUTPUT_DIR)

## 1. 生成/加载合成医疗数据集（任务一 数据源）

确定性种子生成 200 份含噪病历（HTML 标签、全角字符、PII、繁体、重复句），金标准在生成时同步记录。

In [ ]:
from dki.datagen.generate import generate_corpus

if not any(RAW_DATA_DIR.glob("*.txt")):
    stats = generate_corpus(n_docs=200, seed=42)
    print("已生成:", json.dumps(stats, ensure_ascii=False))
else:
    print("已存在", len(list(RAW_DATA_DIR.glob("*.txt"))), "份病历，跳过生成")

# 预览一份含噪原文
sample = sorted(RAW_DATA_DIR.glob("*.txt"))[0].read_text(encoding="utf-8")
print("\n--- 含噪原文（emr_0001）---\n", sample[:300])

## 2. 任务一：数据处理流水线（DataMate 算子清洗）

用 `LocalPipelineRunner`（复刻 DataMate `RayDataset.process` 语义）跑清洗流水线：HTML 去标签 → 全角归一 → 去空格 → PII 脱敏 → 医疗文本清洗。

In [ ]:
from dki.pipeline import LocalPipelineRunner, samples_from_dir
from dki.pipeline.config import load_pipeline_config

process, meta = load_pipeline_config(PROJECT_ROOT / "configs" / "pipeline_medical_etl.yaml")
samples = samples_from_dir(RAW_DATA_DIR, pattern="*.txt")
runner = LocalPipelineRunner()
result = runner.run(process, samples, name="nb-task1-etl")
print(result.report.summary())

# 导出清洗文本（供后续 NER/图谱用）
clean_dir = OUTPUT_DIR / "clean"
clean_dir.mkdir(parents=True, exist_ok=True)
for s in result.ok_samples:
    if s.get("text"):
        (clean_dir / f"{s['fileId']}.txt").write_text(s["text"], encoding="utf-8")

cleaned = (clean_dir / "emr_0001.txt").read_text(encoding="utf-8")
print("\n--- 清洗后（emr_0001，噪声已去除）---\n", cleaned[:300])

## 3. 任务二：知识图谱构建 + 抽取质量评测

实体抽取（疾病/症状/药物/检查/治疗/科室）→ 关系抽取（诊断/治疗/并发…）→ 三元组构建校验 → 跨文档聚合成图谱。抽取三元组 vs 1570 条金标准算 P/R/F1。

In [ ]:
from dki.kg.build import build_kg_from_samples
from dki.kg.eval import evaluate_triples, format_eval_report, predictions_from_samples

process_kg, _ = load_pipeline_config(PROJECT_ROOT / "configs" / "pipeline_kg_build.yaml")
kg_samples = samples_from_dir(clean_dir, pattern="*.txt")
kg_result = runner.run(process_kg, kg_samples, name="nb-task2-kg")

kg = build_kg_from_samples(kg_result.samples)
print("图谱规模:", json.dumps(kg.stats(), ensure_ascii=False))

# 抽取质量 vs 金标
preds = predictions_from_samples(kg_result.samples)
eval_res = evaluate_triples(preds)
print("\n" + format_eval_report(eval_res))

## 4. 任务二：知识图谱问答（无 LLM 模板引擎）

模板问答引擎（正则 + 实体链接 + 反向查询），回答带证据句，无需 LLM。

In [ ]:
from dki.kg.qa import answer_question

for q in ["高血压有哪些症状？", "2型糖尿病有什么并发症？", "冠心病用什么药？"]:
    res = answer_question(kg, q)
    print(f"Q: {q}")
    print(f"A: {res['answer']}")
    if res["evidence"]:
        ev = res["evidence"][0]
        print(f"   证据: [{ev.get('doc','')}] {ev.get('sent','')}")
    print()

## 5. NER 微调（任务一加分：构造数据集 + 微调 <1B 小模型）

用 dict 后端（F1≈99.97%）对清洗文本做**远程监督弱标注**，自动生成 BIO 字符标签，微调 `bert-base-chinese`（101.7M «1B）成真实医疗 NER。

下表为在 **40 份训练不可见的验证集**上的实体级 F1（训练在服务器 CPU 完成，结果落盘 `outputs/ner_finetune/metrics.json`）。

In [ ]:
# 弱标注数据集构建（本地即可跑，纯 python）
from dki.nlp.weak_label import build_ner_dataset
ds_stats = build_ner_dataset(gold_filter=True)
print("弱标注数据集:", json.dumps(ds_stats, ensure_ascii=False, indent=2))

In [ ]:
# 三后端实体级 F1 对比（读服务器训练产物）
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["axes.unicode_minus"] = False

metrics_path = OUTPUT_DIR / "ner_finetune" / "metrics.json"
if metrics_path.exists():
    m = json.loads(metrics_path.read_text(encoding="utf-8"))
    ef = m.get("eval_f1", {})
    rows = [("Fine-tuned BERT", ef.get("finetuned_bert", {})),
            ("dict (teacher)", ef.get("dict", {})),
            ("Untuned BERT", ef.get("untuned_bert", {}))]
    print(f"训练: {m.get('n_train')} 样本 / {m.get('epochs')} epoch / {m.get('train_seconds')}s (CPU)\n")
    labels, f1s = [], []
    for name, r in rows:
        if r:
            print(f"{name:18s} P={r.get('precision')}  R={r.get('recall')}  F1={r.get('f1')}")
            labels.append(name); f1s.append(r.get("f1", 0))
    fig, ax = plt.subplots(figsize=(6, 3.2))
    bars = ax.bar(labels, f1s, color=["#2563eb", "#16a34a", "#dc2626"])
    ax.set_ylabel("Entity-level F1"); ax.set_ylim(0, 1.05)
    ax.set_title("NER backend comparison (held-out 40 docs)")
    for b, v in zip(bars, f1s):
        ax.text(b.get_x() + b.get_width()/2, v + 0.02, f"{v:.3f}", ha="center")
    plt.tight_layout(); plt.show()
else:
    print("未找到 metrics.json；在服务器运行 scripts/train_ner.py 后回传即可。")

## 6. 任务三：数据分析（NL2SQL）

建分析库（装载任务一结构化表 + 任务二图谱）→ 自然语言问题经 NL2SQL 引擎（规则模板 + LLM 兜底）转 SQL 执行。

In [ ]:
from dki.analysis.database import build_database
from dki.analysis.nl2sql import NL2SQL

db_path = OUTPUT_DIR / "insight.db"
build_database(db_path=db_path,
               visits_csv=RAW_DATA_DIR / "structured" / "visits.csv",
               patients_csv=RAW_DATA_DIR / "structured" / "patients.csv",
               kg=kg)
engine = NL2SQL(db_path=db_path)

for q in ["各科室就诊量是多少", "男女患者各有多少人"]:
    r = engine.query(q)
    print(f"Q: {q}")
    print(f"SQL({r.get('engine')}): {r.get('sql')}")
    cols, rows = r.get("columns") or [], r.get("rows") or []
    print("  ", " | ".join(map(str, cols)))
    for row in rows[:6]:
        print("  ", " | ".join(map(str, row)))
    print()

## 7. 任务三：BI 图文报告（ECharts，内联展示）

生成单文件 ECharts HTML 报告（柱状/折线/饼图/图谱力导向图 + 图文解读），内联到 Notebook。

In [ ]:
from dki.analysis.bi import build_default_report
from IPython.display import IFrame, display

report_path = build_default_report(db_path=db_path, kg=kg,
                                   out_path=OUTPUT_DIR / "reports" / "insight_report.html")
print("BI 报告:", report_path)
# 内联展示（ECharts 交互图表 + 图谱力导向图）
rel = os.path.relpath(report_path, Path.cwd())
display(IFrame(src=rel, width="100%", height=650))

## 8. NPU 算子优化（任务二加分：吞吐阶梯 + 过拟合诚实披露）

微调 NER 模型在昇腾 910B2C 上的端到端流水线成绩。四层优化叠加冲极限：
**INT8 量化** + **更小模型(rbt6, 6层)** + **fast-tokenize(offset 下沉)**，
把吞吐从 bert-FP16 的 2516 推到 rbt6-INT8-fast 的 ~5961 样本/s（2.37×）。
详见 `docs/npu_optimization.md` §5.7。

In [ ]:
# NPU 优化吞吐阶梯：bert/rbt6 × FP16/INT8 × fast-tokenize（读服务器 bench 产物）
import matplotlib.pyplot as plt

def _tp(sub, name):
    p = OUTPUT_DIR / 'bench' / sub / name
    if p.exists():
        d = json.loads(p.read_text(encoding='utf-8'))
        return d.get('throughput_samples_s')
    return None

def _f1(sub, name):
    p = OUTPUT_DIR / 'bench' / sub / name
    if p.exists():
        return json.loads(p.read_text(encoding='utf-8'))['entity_f1']['f1']
    return None

ladder = [
    ('bert-FP16', _tp('npu_int8','pipeline_ft_fp16_bs64.json'), _f1('npu_int8','eval_om_fp16b.json')),
    ('bert-INT8', _tp('npu_int8','pipeline_ft_int8_bs64.json'), _f1('npu_int8','eval_om_int8.json')),
    ('rbt6-FP16', _tp('npu_rbt6','pipeline_rbt6_fp16_bs64.json'), _f1('npu_rbt6','eval_om_rbt6_fp16.json')),
    ('rbt6-INT8', _tp('npu_rbt6','pipeline_rbt6_int8_bs64.json'), _f1('npu_rbt6','eval_om_rbt6_int8.json')),
    ('rbt6-INT8-fast', _tp('npu_rbt6','pipeline_rbt6_int8_fast_bs64.json'), _f1('npu_rbt6','eval_om_rbt6_int8.json')),
]
ladder = [x for x in ladder if x[1] is not None]
base = ladder[0][1] if ladder else 1
print(f"{'配置':16s} {'吞吐(样本/s)':>12s} {'相对bert-FP16':>12s} {'OM F1':>8s}")
for name, tp, f1 in ladder:
    print(f'{name:16s} {tp:>12.0f} {tp/base:>11.2f}x {(f1 if f1 else float("nan")):>8.4f}')

if ladder:
    fig, ax = plt.subplots(figsize=(8, 3.4))
    names = [x[0] for x in ladder]; tps = [x[1] for x in ladder]
    bars = ax.bar(names, tps, color=['#2563eb','#3b82f6','#16a34a','#f59e0b','#dc2626'][:len(ladder)])
    ax.set_ylabel('samples/s'); ax.set_title('NPU 910B2C throughput ladder (bs64/seq256)')
    for b,v in zip(bars,tps): ax.text(b.get_x()+b.get_width()/2, v+40, f'{v:.0f}', ha='center', fontsize=9)
    plt.xticks(rotation=15); plt.tight_layout(); plt.show()


## 9. 过拟合诊断：分布外（OOD）鲁棒性

train/eval 同源于合成生成器 + 词典，held-out F1 测的是**同分布**泛化。用外部 LLM
生成表述迥异的分布外病历 + LLM 独立标注（逐字 mention），用 **span-overlap** 匹配
测真实鲁棒性——诚实披露边界，并证明微调相对纯词典法的泛化价值
（详见 `docs/ner_finetuning.md` §8）。

In [ ]:
# OOD 鲁棒性：开发集、盲测-2 与预先冻结后生成的 final3（读评测产物）
base_path = OUTPUT_DIR / 'ner_finetune' / 'ood_eval.json'
quality_path = OUTPUT_DIR / 'ner_finetune' / 'ood_eval_quality_calibrated.json'
final2_path = OUTPUT_DIR / 'ner_finetune' / 'ood_final2_quality_v2.json'
final3_path = OUTPUT_DIR / 'ner_finetune' / 'ood_final3_quality_v2.json'
if base_path.exists():
    ood = json.loads(base_path.read_text(encoding='utf-8'))
    if quality_path.exists():
        quality = json.loads(quality_path.read_text(encoding='utf-8'))
        ood['quality'] = quality.get('quality')
    print('既有 OOD 开发集:', ood.get('n_docs'), '篇 /', ood.get('n_gold_entities'), '实体')
    print('%-16s %8s %8s %8s' % ('后端','OOD-P','OOD-R','OOD-F1'))
    for k in ['dict','bert','rbt6','rbt3','quality']:
        r = ood.get(k)
        if r:
            print('%-16s %8.3f %8.3f %8.3f' % (k, r['precision'], r['recall'], r['f1']))
    print('注：quality 规则在诊断该集合后形成，因此 0.952 recall 只属于校准开发成绩。')

    for title, path, keys in [
        ('盲测-2（quality v2 首次外部验收）', final2_path, ['baseline','quality']),
        ('final3（模型/规则预先冻结后生成；只评一次）', final3_path,
         ['dict','baseline','quality','quality_recall']),
    ]:
        if not path.exists():
            continue
        result = json.loads(path.read_text(encoding='utf-8'))
        print(f"\n{title}: {result.get('n_docs')} 篇 / {result.get('n_gold_entities')} 可定位实体")
        for key in keys:
            r = result.get(key)
            if r:
                print('%-16s %8.3f %8.3f %8.3f' % (key, r['precision'], r['recall'], r['f1']))
    print('\n结论：v2 提高盲测 F1；final3 recall mode 的 F1=0.789，但未知主题召回仍未稳定达到 0.9，不作达标宣称。')
else:
    print('未找到 ood_eval.json；见 scripts/gen_ood_eval.py + scripts/eval_ood.py。')

## 10. 权威证据：validator-passing typed claims

上面各节直接读原始 JSON 便于交互展示；本节则加载**机器可校验的 typed claims**
（`outputs/claims/headline_claims.json`，由 `scripts/build_evidence_claims.py` 生成、
`scripts/validate_evidence.py` 校验）。每条 claim 携带枚举化的 `dataset_role /
metric_kind / runtime_kind / evidence_level / promotion_state` 与逐工件 SHA-256，
`load_validated_claims` 在加载时重算哈希，任一不匹配即报错——因此这里显示的数字
一定与磁盘证据一致，且能区分 development/blind、exact/relaxed、probe/formal、cpu/npu。

In [ ]:
from dki.evidence import load_validated_claims

claims_path = OUTPUT_DIR / 'claims' / 'headline_claims.json'
if claims_path.exists():
    claims = load_validated_claims(claims_path, ROOT)  # fail-closed: 哈希不符即抛错
    print(f"已校验 {len(claims)} 条 typed claim（枚举齐全 + 工件 SHA-256 匹配）\n")
    print('%-34s %-20s %-11s %-10s %s' % ('claim_id', 'dataset_role', 'runtime', 'promotion', 'metric'))
    for c in claims:
        m = c.get('metric', {})
        val = m.get('value')
        val_s = (f"{val:.4f}" if isinstance(val, float) else str(val))
        print('%-34s %-20s %-11s %-10s %s=%s' % (
            c['claim_id'], c['dataset_role'], c['runtime_kind'],
            c['promotion_state'], m.get('name', ''), val_s))
    print('\n口径提醒：promotion_state=candidate/shadow 的 NPU 项为目标机候选证据；'
          '\ndataset_role=historical_blind 的 final3 为一次性封存盲测，未达未知分布 recall>=0.90。')
else:
    print('未找到 claims，请先运行： python scripts/build_evidence_claims.py')


## 小结

- **数据 → 知识 → 洞察闭环**：清洗 → 图谱(F1≈99.97%) → 问答 → NL2SQL(100%) → BI 报告，全程可复现、零网络依赖。
- **真实微调 NER**：dict 远程监督微调 bert-base-chinese，验证集 F1≈0.99，补齐任务一「微调 <1B 小模型」加分。
- **INT8 极致性能**：昇腾 910B2C 上 INT8 相对 FP16 吞吐 1.26×、能效 1.46×，实体 F1 仅降 0.81 点——加速挂到真实医疗精度。

详见 `README.md`、`docs/npu_optimization.md`、`docs/ner_finetuning.md`。